
# Lab Session: Easy Seq2Seq Model for Text-to-Text Learning



## 1. Import required libraries



In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0



## 2. Create a tiny dataset

We use a very small text mapping dataset.

### Idea
The model learns:
- input sequence → output sequence

### Example
- `hi` → `hola`
- `bye` → `adios`

This is only for understanding the Seq2Seq workflow, not for building a real translator.


In [2]:
input_texts = ["hi", "hello", "bye", "thanks", "yes", "no",
               "good", "welcome", "please", "sorry",
               "morning", "night", "friend", "love", "help"]

target_texts = ["hola", "hola", "adios", "gracias", "si", "no",
                "bueno", "bienvenido", "por favor", "lo siento",
                "manana", "noche", "amigo", "amor", "ayuda"]

print("Input texts :", input_texts)
print("Target texts:", target_texts)


Input texts : ['hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'welcome', 'please', 'sorry', 'morning', 'night', 'friend', 'love', 'help']
Target texts: ['hola', 'hola', 'adios', 'gracias', 'si', 'no', 'bueno', 'bienvenido', 'por favor', 'lo siento', 'manana', 'noche', 'amigo', 'amor', 'ayuda']



## 3. Add special start and end symbols

In Seq2Seq models, the decoder needs:
- a **start token** to know when to begin generating output
- an **end token** to know when to stop

Here we use:
- `\t` as start token
- `\n` as end token


In [3]:

target_texts = ["\t" + text + "\n" for text in target_texts]

print("Target texts with start/end symbols:")
for t in target_texts:
    print(repr(t))


Target texts with start/end symbols:
'\thola\n'
'\thola\n'
'\tadios\n'
'\tgracias\n'
'\tsi\n'
'\tno\n'
'\tbueno\n'
'\tbienvenido\n'
'\tpor favor\n'
'\tlo siento\n'
'\tmanana\n'
'\tnoche\n'
'\tamigo\n'
'\tamor\n'
'\tayuda\n'



## 4. Build character vocabulary

We perform **character-level encoding** instead of word-level encoding because it is easier for a small lab example.

This means each character gets an integer index.


In [4]:

input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))

input_char_to_index = {char: i + 1 for i, char in enumerate(input_chars)}
target_char_to_index = {char: i + 1 for i, char in enumerate(target_chars)}

input_index_to_char = {i: char for char, i in input_char_to_index.items()}
target_index_to_char = {i: char for char, i in target_char_to_index.items()}

num_encoder_tokens = len(input_chars) + 1
num_decoder_tokens = len(target_chars) + 1

print("Input characters :", input_chars)
print("Target characters:", target_chars)
print("Number of encoder tokens:", num_encoder_tokens)
print("Number of decoder tokens:", num_decoder_tokens)


Input characters : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'v', 'w', 'y']
Target characters: ['\t', '\n', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y']
Number of encoder tokens: 22
Number of decoder tokens: 24



## 5. Find maximum sequence lengths

Since all sequences in a batch must have the same length, we identify:
- maximum encoder input length
- maximum decoder input length


In [5]:

max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)


Max encoder sequence length: 7
Max decoder sequence length: 12



## 6. Prepare encoder input, decoder input, and decoder output

### Encoder input
Stores the input sequence indices.

### Decoder input
Starts with the start token and then receives target characters.

### Decoder output
The model is trained to predict the **next** character in the target sequence.

This shifting is very important in Seq2Seq training.


In [6]:

encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length), dtype="int32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length), dtype="int32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[i, t] = input_char_to_index[char]

    # Decoder input and decoder target
    for t, char in enumerate(target_text):
        decoder_input_data[i, t] = target_char_to_index[char]
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_index[char]] = 1.0

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


Encoder input shape: (15, 7)
Decoder input shape: (15, 12)
Decoder target shape: (15, 12, 24)



## 7. Build the encoder

The encoder reads the input sequence and produces internal states:
- hidden state (`state_h`)
- cell state (`state_c`)

These states act as the information passed to the decoder.


In [10]:

latent_dim = 256

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=num_encoder_tokens, output_dim=32, mask_zero=True, name="encoder_embedding")(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True, dropout=0.1, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]



## 8. Build the decoder

The decoder:
- takes the previous output token
- uses the encoder states as initial context
- predicts the next token at each time step


In [11]:

decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=16, mask_zero=True, name="decoder_embedding")
decoder_embedding = decoder_embedding_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, dropout=0.1, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)



## 9. Define the full Seq2Seq model

The full training model takes:
- encoder input sequence
- decoder input sequence

and learns to predict:
- decoder output sequence


In [16]:

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 32)  │        704 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 16)  │        384 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    295,936 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal_3[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    279,552 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 24)  │      6,168 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 582,744 (2.22 MB)

 Trainable params: 582,744 (2.22 MB)

 Non-trainable params: 0 (0.00 B)


## 10. Train the model

Because the dataset is very small, we train for more epochs so that the model can learn the tiny mappings well.


In [17]:

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=1,
    epochs=300,
    verbose=1
)


Epoch 1/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.8661 - loss: 0.0042
Epoch 2/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8482 - loss: 0.0367
Epoch 3/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.8661 - loss: 0.0058
Epoch 4/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.8661 - loss: 0.0051
Epoch 5/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8661 - loss: 0.0041
Epoch 6/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8661 - loss: 0.0044
Epoch 7/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8661 - loss: 0.0042
Epoch 8/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.8661 - loss: 0.0037
Epoch 9/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8661 - loss: 0.0036
Epoch 10/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8661 - loss: 0.0036
Epoch 11/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8661 - loss: 0.0038
Epoch 12/300
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


## 11. Build inference models for prediction

During training, the full target sequence is known.

During testing, we generate one character at a time:
1. encode the input
2. start decoder with the start token
3. predict next character
4. feed predicted character back to decoder
5. stop when end token is produced


In [18]:

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)



## 12. Define decoding function

This function converts a new input text into an output text using the trained encoder and decoder.


In [19]:

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1), dtype="int32")
    target_seq[0, 0] = target_char_to_index["\t"]

    decoded_sentence = ""

    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_char.get(sampled_token_index, "")

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1), dtype="int32")
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence



## 13. Test the Seq2Seq model

Now we check whether the model has learned the simple sequence mapping.


In [20]:

for seq_index in range(len(input_texts)):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print(f"Input: {input_texts[seq_index]}  -->  Predicted Output: {decoded_sentence}")


Input: hi  -->  Predicted Output: hola
Input: hello  -->  Predicted Output: hola
Input: bye  -->  Predicted Output: adios
Input: thanks  -->  Predicted Output: gracias
Input: yes  -->  Predicted Output: si
Input: no  -->  Predicted Output: no
Input: good  -->  Predicted Output: bueno
Input: welcome  -->  Predicted Output: bienvenido
Input: please  -->  Predicted Output: por favor
Input: sorry  -->  Predicted Output: lo siento
Input: morning  -->  Predicted Output: manana
Input: night  -->  Predicted Output: noche
Input: friend  -->  Predicted Output: amigo
Input: love  -->  Predicted Output: amor
Input: help  -->  Predicted Output: ayuda



## 14. Try your own input

This helper function allows students to test a word from the known training vocabulary.


In [21]:

def encode_input_text(text):
    seq = np.zeros((1, max_encoder_seq_length), dtype="int32")
    for t, char in enumerate(text[:max_encoder_seq_length]):
        if char in input_char_to_index:
            seq[0, t] = input_char_to_index[char]
    return seq

# Examples
test_words = ["hi", "bye", "yes", "no", "hello", "thanks"]
for word in test_words:
    encoded = encode_input_text(word)
    print(f"Input: {word}  -->  Output: {decode_sequence(encoded)}")


Input: hi  -->  Output: hola
Input: bye  -->  Output: adios
Input: yes  -->  Output: si
Input: no  -->  Output: no
Input: hello  -->  Output: hola
Input: thanks  -->  Output: gracias
